Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
Data Science - sección 40

Cristian Túnchez (231359)  
Nadissa López (23764)

# Laboratorio 4 (Parte 2) — Modelos de Machine Learning

Este notebook cubre los **ejercicios 4 al 10** de la segunda parte del laboratorio:

4. Construcción de tres modelos de clasificación.
5. Evaluación y comparación de los modelos.
6. Validación espacial por bloques y validación temporal.
7. Generalización entre lagos.
8. Interpretación y explicabilidad con SHAP.
9. Generación de mapas predictivos.
10. Análisis y conclusiones.

Inicia a patrir del conjunto de datos construido en `03_preparacion_dataset_ml.ipynb` y guardado en `data/processed/dataset_ml.csv`, documentado en `codebook.md`.

**Preparación del entorno**

In [1]:
import sys
import warnings
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from matplotlib.colors import BoundaryNorm, ListedColormap
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# La carpeta src/ contiene los scripts
sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import dataset_ml as dml
import mapas as mp
import procesar

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

SEMILLA = 42
LAGOS = list(config.FECHAS)
COLOR = {"Atitlan": "#1f77b4", "Amatitlan": "#d62728"}

datos = dml.cargar_dataset()
print("Observaciones:", f"{len(datos):,}")
print("Predictores  :", len(dml.PREDICTORES))
print("Prevalencia  :", f"{datos['y_alta'].mean() * 100:.3f} %")

Observaciones: 401,656
Predictores  : 13
Prevalencia  : 1.213 %


---

## Ejercicio 4 — Construcción de modelos de Machine Learning

### 4.2 División de los datos

Se separa el 70% para entrenamiento y el 30% para prueba, con **estratificación** por la variable respuesta. La estratificación es indispensable, con una prevalencia del 1.21%, una división puramente aleatoria podría dejar muy pocos positivos en alguno de los dos conjuntos.

Este conjunto de prueba se guarda y **se reutiliza sin modificar** en todo el ejercicio 5, para que las comparaciones entre modelos sean justas.

In [2]:
X = datos[dml.PREDICTORES]
y = datos["y_alta"]

X_ent, X_prueba, y_ent, y_prueba = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEMILLA
)
# Los identificadores se dividen con los mismos indices, para poder rastrear cada
# observacion de prueba hasta su lago, fecha y posicion.
meta_prueba = datos.loc[X_prueba.index, ["lago", "fecha", "fila", "columna", "bloque_1km", "chla"]]

particion = pd.DataFrame(
    {
        "observaciones": [len(X_ent), len(X_prueba)],
        "positivos": [int(y_ent.sum()), int(y_prueba.sum())],
        "prevalencia_pct": [round(y_ent.mean() * 100, 4), round(y_prueba.mean() * 100, 4)],
    },
    index=["entrenamiento (70 %)", "prueba (30 %)"],
)
particion

,observaciones,positivos,prevalencia_pct
entrenamiento (70 %),281159,3411,1.2132
prueba (30 %),120497,1462,1.2133


### 4.1 y 4.3 Definición de los modelos y ajuste de hiperparámetros

Se construyen los tres modelos solicitados. Los tres reciben `class_weight="balanced"`, que pondera cada clase de forma inversamente proporcional a su frecuencia; sin eso, la función de pérdida quedaría dominada por el 98.79% de negativos.

| Modelo | Por qué se incluye | Hiperparámetros ajustados |
| --- | --- | --- |
| **Regresión Logística** | Modelo lineal de referencia. Sirve de base, si un modelo complejo no lo supera, la complejidad no aporta. Necesita `StandardScaler` porque las variables viven en escalas muy distintas (reflectancias de 0 a 1 frente a distancias de hasta 3 660 m). | `C`, la inversa de la fuerza de regularización. Importa por la fuerte colinealidad detectada en el ejercicio 1.5. |
| **Random Forest** | Captura interacciones y relaciones no lineales, y tolera bien la redundancia entre variables. | `max_depth` y `min_samples_leaf`. Se regulariza desde el inicio. |
| **Gradient Boosting** | `HistGradientBoostingClassifier`, la implementación por histogramas de scikit-learn. Es el equivalente a LightGBM dentro de la librería. | `learning_rate` y `max_leaf_nodes`, que gobiernan el compromiso entre velocidad de aprendizaje y complejidad de cada árbol. |

El criterio de selección es la **precisión promedio** (`average_precision`, equivalente al área bajo la curva de precisión-recall). Se prefiere al accuracy y al ROC-AUC porque es la métrica que no se deja engañar por el desbalance. La validación cruzada interna usa 3 particiones estratificadas.

In [3]:
cv_interna = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEMILLA)

configuraciones = {
    "Regresion Logistica": (
        make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEMILLA),
        ),
        {"logisticregression__C": [0.1, 1.0, 10.0]},
    ),
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=100, class_weight="balanced", n_jobs=2, random_state=SEMILLA
        ),
        {"max_depth": [10, 15], "min_samples_leaf": [20, 50]},
    ),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(class_weight="balanced", random_state=SEMILLA),
        {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [31, 63]},
    ),
}

modelos = {}
busquedas = []
for nombre, (estimador, rejilla) in configuraciones.items():
    busqueda = GridSearchCV(
        estimador, rejilla, scoring="average_precision", cv=cv_interna, n_jobs=1, refit=True
    )
    busqueda.fit(X_ent, y_ent)
    modelos[nombre] = busqueda.best_estimator_
    busquedas.append(
        {
            "modelo": nombre,
            "combinaciones": len(busqueda.cv_results_["params"]),
            "mejores_hiperparametros": busqueda.best_params_,
            "pr_auc_validacion": round(busqueda.best_score_, 4),
        }
    )
    print(f"[ok] {nombre}: {busqueda.best_params_}  PR-AUC(cv) = {busqueda.best_score_:.4f}")

pd.DataFrame(busquedas)

[ok] Regresion Logistica: {'logisticregression__C': 0.1}  PR-AUC(cv) = 0.9220


[ok] Random Forest: {'max_depth': 15, 'min_samples_leaf': 20}  PR-AUC(cv) = 0.9671


[ok] Gradient Boosting: {'learning_rate': 0.1, 'max_leaf_nodes': 31}  PR-AUC(cv) = 0.9706


,modelo,combinaciones,mejores_hiperparametros,pr_auc_validacion
0,Regresion Logistica,3,{'logisticregression__C': 0.1},0.9220
1,Random Forest,4,"{'max_depth': 15, 'min_samples_leaf': 20}",0.9671
2,Gradient Boosting,4,"{'learning_rate': 0.1, 'max_leaf_nodes': 31}",0.9706


Las tres búsquedas eligieron la configuración **más simple o más regularizada** de las que se probaron:

| Modelo | Elegido | PR-AUC (validación interna) |
| --- | --- | --- |
| Regresión Logística | `C = 0.1` | 0.9220 |
| Random Forest | `max_depth = 15`, `min_samples_leaf = 20` | 0.9671 |
| Gradient Boosting | `learning_rate = 0.1`, `max_leaf_nodes = 31` | 0.9706 |

Que la Regresión Logística prefiera `C = 0.1`, el valor con regularización más fuerte del conjunto evaluado, es coherente con la colinealidad detectada en el ejercicio 1.5; con 27 pares de variables correlacionadas por encima de 0.9, la penalización fuerte estabiliza los coeficientes. El Gradient Boosting eligió `max_leaf_nodes = 31` en lugar de 63, es decir, árboles más pequeños, lo que sugiere que la relación entre las bandas y la floración no requiere interacciones muy profundas.

In [4]:
print("Verificacion de que el conjunto de prueba es unico y no se toco:")
print("  observaciones de prueba:", f"{len(X_prueba):,}")
print("  positivos              :", int(y_prueba.sum()))
print("  suma de control (hash) :", pd.util.hash_pandas_object(X_prueba.index).sum())
print("\nLos tres modelos se evaluaran sobre exactamente estas observaciones.")

Verificacion de que el conjunto de prueba es unico y no se toco:
  observaciones de prueba: 120,497
  positivos              : 1462
  suma de control (hash) : 944216947976199151

Los tres modelos se evaluaran sobre exactamente estas observaciones.
